# Silver Layer: Cleaned & Incremental Processing

**Purpose:** Transform bronze tables into cleaned silver tables with deduplication and business logic

**Environment:** Parameterized for dev/uat/prod deployment

**Features:**
- **Deduplication** using primary keys and timestamps
- **Incremental updates** based on `updated_at` timestamps
- **Business transformations** and data quality improvements

**CI/CD Ready:** Uses environment parameter for multi-environment execution

In [0]:
# Environment parameter (created via widget)
dbutils.widgets.dropdown("environment", "dev", ["dev", "uat", "prod"], "Environment")
env = dbutils.widgets.get("environment")

# Set catalog name based on environment
catalog = f"1_{env}"

# Print configuration
print(f"="*60)
print(f"SILVER LAYER CONFIGURATION")
print(f"="*60)
print(f"Environment: {env}")
print(f"Catalog: {catalog}")
print(f"Source: {catalog}.bronze")
print(f"Target: {catalog}.silver")
print(f"="*60)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col, trim, upper, when, current_timestamp

# Read bronze customers from environment-specific catalog
df_customers_bronze = spark.table(f"{catalog}.bronze.customers")

# Deduplication: Keep latest record per customer_id based on updated_at
window_spec = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())

df_customers_silver = df_customers_bronze \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \
    .withColumn("name", trim(col("name"))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("state", upper(trim(col("state")))) \
    .withColumn("processed_at", current_timestamp())

# Write to silver layer in environment-specific catalog
df_customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog}.silver.customers")

print(f"✓ Processed {df_customers_silver.count():,} customers to {catalog}.silver.customers")

In [0]:
# Read bronze orders from environment-specific catalog
df_orders_bronze = spark.table(f"{catalog}.bronze.orders")

# Deduplication: Keep latest record per order_id based on updated_at
window_spec = Window.partitionBy("order_id").orderBy(col("updated_at").desc())

df_orders_silver = df_orders_bronze \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \
    .withColumn("order_status", upper(trim(col("order_status")))) \
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)")) \
    .withColumn("processed_at", current_timestamp()) \
    .filter(col("order_id").isNotNull()) \
    .filter(col("customer_id").isNotNull())

# Write to silver layer in environment-specific catalog
df_orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog}.silver.orders")

print(f"✓ Processed {df_orders_silver.count():,} orders to {catalog}.silver.orders")

In [0]:
# Read bronze order_items from environment-specific catalog
df_order_items_bronze = spark.table(f"{catalog}.bronze.order_items")

# Deduplication: Keep latest record per order_item_id based on updated_at
window_spec = Window.partitionBy("order_item_id").orderBy(col("updated_at").desc())

df_order_items_silver = df_order_items_bronze \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \
    .withColumn("price", col("price").cast("decimal(10,2)")) \
    .withColumn("line_total", col("quantity") * col("price")) \
    .withColumn("processed_at", current_timestamp()) \
    .filter(col("order_item_id").isNotNull()) \
    .filter(col("order_id").isNotNull()) \
    .filter(col("quantity") > 0)

# Write to silver layer in environment-specific catalog
df_order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog}.silver.order_items")

print(f"✓ Processed {df_order_items_silver.count():,} order items to {catalog}.silver.order_items")

In [0]:
# Read bronze products from environment-specific catalog
df_products_bronze = spark.table(f"{catalog}.bronze.products")

# Deduplication: Keep latest record per product_id based on updated_at
window_spec = Window.partitionBy("product_id").orderBy(col("updated_at").desc())

df_products_silver = df_products_bronze \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("category", upper(trim(col("category")))) \
    .withColumn("price", col("price").cast("decimal(10,2)")) \
    .withColumn("processed_at", current_timestamp()) \
    .filter(col("product_id").isNotNull()) \
    .filter(col("product_name").isNotNull())

# Write to silver layer in environment-specific catalog
df_products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog}.silver.products")

print(f"✓ Processed {df_products_silver.count():,} products to {catalog}.silver.products")

In [0]:
# Verify all silver tables were created
silver_tables = spark.sql(f"SHOW TABLES IN {catalog}.silver").toPandas()
print(f"Silver Layer Tables in {catalog}.silver:")
display(silver_tables)

# Show record counts for each silver table
print(f"\nRecord Counts in {catalog}.silver:")
for table in ['customers', 'orders', 'order_items', 'products']:
    count = spark.table(f"{catalog}.silver.{table}").count()
    print(f"  {table}: {count:,} records")